# NB2b — Ablations (multi-seed, self-contained)

Runs the full model + 5 ablations, **3 seeds each**, and reports val IoU as
mean ± std so differences smaller than seed-noise are visibly non-significant.

This notebook is standalone — do NOT paste the single-run training cells before
it. Attach `lgmf_c` (v4), Sen1Floods11 imagery, `lgmf-index`. Internet: On.

Runtime ~3h (18 runs x 45 ep). A resume guard skips any (config, seed) already
done, so a re-run continues rather than restarting.


In [ ]:
import sys, os, glob, json, time, random, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
assert lgmf.__version__ >= "4.0.0"

hits = glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)
split_dir = os.path.dirname(hits[0]); stats = json.load(open(f"{split_dir}/norm_stats.json"))
print("index:", split_dir, "| split:", stats["split_source"])

## Config: seeds, epochs, ablation matrix

In [ ]:
SEEDS  = [42, 1, 2]           # 3 seeds -> mean +/- std
EPOCHS = 45                   # best.pt selection captures the peak; extra epochs cost only time
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"
regs = ["both", "s1_only", "s2_only"]

# (name, overrides on the locked v4_ft config)
ABLATIONS = [
    ("full",    {}),                              # locked paper model
    ("no_film", {"film_on": False}),
    ("concat",  {"gate_mode": "concat"}),
    ("sum",     {"gate_mode": "sum"}),
    ("uniform", {"use_curriculum": False}),
    ("no_cons", {"consistency_weight": 0.0}),
]

def make_cfg(name, seed, **ov):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type, c.stem_stride = 0.99, "focal_tversky", 2   # locked
    c.split_dir, c.ckpt_dir = split_dir, "/kaggle/working/ckpt"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, name
    for k, v in ov.items(): setattr(c, k, v)
    return c

## Determinism controls + single-run trainer

In [ ]:
def seed_all(sd):
    random.seed(sd); np.random.seed(sd)
    torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False          # stop kernel autotune drift

def worker_init(wid):
    s = (torch.initial_seed() + wid) % (2**32)      # pin augmentation RNG per worker
    np.random.seed(s); random.seed(s)

def run_one(cfg):
    seed_all(cfg.seed)
    g = torch.Generator(); g.manual_seed(cfg.seed)   # pin shuffling order
    tr_ds = Sen1Floods11(cfg, f"{split_dir}/train_index.json", augment=True)
    va_ds = Sen1Floods11(cfg, f"{split_dir}/val_index.json",   augment=False)
    tr = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                    pin_memory=pin, drop_last=True, generator=g, worker_init_fn=worker_init)
    va = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False,
                    num_workers=cfg.num_workers, pin_memory=pin)
    model = build_model(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type == "cuda") else None
    ema = ModelEMA(model, cfg.ema_decay) if cfg.use_ema else None
    em = ema.module if ema else model
    best, best_val = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            best_val = {r: {"iou": val[r]["iou"], "f1": val[r]["f1"],
                            "flood_only_iou": val[r].get("flood_only_iou")} for r in regs}
            D = os.path.join(cfg.ckpt_dir, cfg.run_name, f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(),
                        "ema": ema.module.state_dict() if ema else None,
                        "epoch": epoch, "val": best_val, "cfg": dict(cfg.__dict__)},
                       os.path.join(D, "best.pt"))
    return best, best_val

## Run all (config x seed) with resume guard

In [ ]:
results = {name: [] for name, _ in ABLATIONS}
t0 = time.time()
for name, ov in ABLATIONS:
    for sd in SEEDS:
        D = os.path.join("/kaggle/working/ckpt", name, f"seed{sd}")
        rj = os.path.join(D, "result.json")
        if os.path.exists(rj):
            rec = json.load(open(rj)); best, bv = rec["best"], rec["val"]
            print(f"[skip] {name:8s} seed{sd}  mean {best:.3f}")
        else:
            best, bv = run_one(make_cfg(name, sd, **ov))
            os.makedirs(D, exist_ok=True)
            json.dump({"best": best, "val": bv}, open(rj, "w"), indent=2, default=str)
            print(f"[done] {name:8s} seed{sd}  mean {best:.3f}   "
                  f"both {bv['both']['iou']:.3f} s1 {bv['s1_only']['iou']:.3f} "
                  f"s2 {bv['s2_only']['iou']:.3f}   ({(time.time()-t0)/60:.0f} min)")
        results[name].append({"seed": sd, "best": best, "val": bv})
print(f"\nall runs done in {(time.time()-t0)/60:.0f} min")

## Aggregate: mean ± std, with significance flag

In [ ]:
def ious(name, reg): return np.array([r["val"][reg]["iou"] for r in results[name]])
def means(name):     return np.array([r["best"] for r in results[name]])

agg = {}
for name, _ in ABLATIONS:
    agg[name] = {reg: (float(ious(name, reg).mean()), float(ious(name, reg).std())) for reg in regs}
    mm = means(name); agg[name]["mean"] = (float(mm.mean()), float(mm.std()))
json.dump({"seeds": SEEDS, "epochs": EPOCHS, "agg": agg, "raw": results},
          open("/kaggle/working/ablation_summary.json", "w"), indent=2, default=str)

full_m, full_s = agg["full"]["mean"]
def cell(a, reg): m, s = a[reg]; return f"{m:.3f}±{s:.3f}"
print("=" * 92)
print(f"{'config':10s} {'both':>13s} {'s1_only':>13s} {'s2_only':>13s} {'mean':>13s}   {'d(mean)':>9s}")
print("-" * 92)
for name, _ in ABLATIONS:
    a = agg[name]
    if name == "full":
        d = "  (ref)"
    else:
        dm = a["mean"][0] - full_m
        sig = "*" if abs(dm) > max(full_s, a["mean"][1]) else " "
        d = f"{dm:+.3f}{sig}"
    print(f"{name:10s} {cell(a,'both'):>13s} {cell(a,'s1_only'):>13s} "
          f"{cell(a,'s2_only'):>13s} {cell(a,'mean'):>13s}   {d:>9s}")
print("=" * 92)
print("* = |d(mean)| exceeds the larger seed-std (indicative; n=3 is too small for a formal test)")
print("Read regimes, not just mean: FiLM/gate act on single-modality; NB4's degradation curve is the real test of the gate.")